# Validação de Fusão — Classificação Binária por Quadrante (OvR)

Este notebook avalia empiricamente se as features de fingerprint acústico **complementam** o openSMILE
na detecção binária de cada quadrante emocional (Q1–Q4 no espaço Valence × Arousal).

Cada quadrante é tratado como **classe positiva** em um problema binário independente (one-vs-rest):
- **Q1** vs resto — High Valence / High Arousal
- **Q2** vs resto — Low Valence  / High Arousal
- **Q3** vs resto — Low Valence  / Low Arousal
- **Q4** vs resto — High Valence / Low Arousal

> **Nota metodológica:** o objetivo *não é provar previamente* que a fusão melhora,
> mas medir empiricamente se há ganho real, em quais quadrantes e sob quais cenários.

**Validação oficial:** `GroupKFold(n_splits=5)` agrupado por `song_id`.  
**Métrica principal:** `positive_f1` (F1 da classe positiva — o quadrante alvo).

## 1. Imports e configuração

In [1]:
import os
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 1))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display, Markdown
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_HGBC = True
except ImportError:
    HAS_HGBC = False

print(f'pandas {pd.__version__} | numpy {np.__version__}')
print(f'HistGradientBoostingClassifier: {HAS_HGBC}')

pandas 2.1.4 | numpy 1.26.4
HistGradientBoostingClassifier: True


In [2]:
@dataclass
class Config:
    project_dir: Path = Path(r'C:\dev\python\TCC Ajustado')
    random_state: int = 42
    n_splits: int = 5
    n_estimators: int = 150
    n_jobs: int = -1
    selector_k_values: List[int] = field(default_factory=lambda: [30, 60, 100, 200])
    use_raw_peaks: bool = True
    export_html: bool = True
    show_figures: bool = True

    QUADRANT_ORDER: List[str] = field(default_factory=lambda: [
        'Q1_High_Valence_High_Arousal',
        'Q2_Low_Valence_High_Arousal',
        'Q3_Low_Valence_Low_Arousal',
        'Q4_High_Valence_Low_Arousal',
    ])
    QUADRANT_SHORT: Dict[str, str] = field(default_factory=lambda: {
        'Q1_High_Valence_High_Arousal': 'Q1',
        'Q2_Low_Valence_High_Arousal':  'Q2',
        'Q3_Low_Valence_Low_Arousal':   'Q3',
        'Q4_High_Valence_Low_Arousal':  'Q4',
    })
    QUADRANT_COLORS: Dict[str, str] = field(default_factory=lambda: {
        'Q1_High_Valence_High_Arousal': '#2E7D32',
        'Q2_Low_Valence_High_Arousal':  '#C62828',
        'Q3_Low_Valence_Low_Arousal':   '#1565C0',
        'Q4_High_Valence_Low_Arousal':  '#F9A825',
    })

    META_COLS: List[str] = field(default_factory=lambda: [
        'song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec',
        'block_duration_sec', 'valence', 'arousal', 'quadrant', 'quadrant_label',
        'quadrant_label_std', 'method', 'title', 'artist', 'genre',
        'band_name', 'band_id', 'band_low_hz', 'band_high_hz', 'topk_per_band',
    ])

    def data_dir(self) -> Path:
        return self.project_dir / 'Dados'

    def output_dir(self) -> Path:
        p = self.project_dir / 'Dados' / 'pycaret_fusion_binary_by_quadrant_validation'
        p.mkdir(parents=True, exist_ok=True)
        return p

    def figures_dir(self) -> Path:
        p = self.output_dir() / 'figures'
        p.mkdir(parents=True, exist_ok=True)
        return p


cfg = Config()
QUADRANT_COL = 'quadrant_label'
JOIN_KEYS    = ['song_id', 'block_start_sec', 'block_end_sec']
print('Output dir:', cfg.output_dir())
print('Figures dir:', cfg.figures_dir())

Output dir: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_binary_by_quadrant_validation
Figures dir: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_binary_by_quadrant_validation\figures


## 2. Funções utilitárias

In [3]:
def numeric_valid_cols(df: pd.DataFrame, exclude: List[str]) -> List[str]:
    excl = set(exclude)
    out = []
    for col in df.columns:
        if col in excl:
            continue
        s = pd.to_numeric(df[col], errors='coerce')
        if s.notna().sum() > 0 and s.nunique(dropna=True) > 1:
            out.append(col)
    return out


def round_times(df: pd.DataFrame, decimals: int = 1) -> pd.DataFrame:
    for col in ['block_start_sec', 'block_end_sec']:
        if col in df.columns:
            df[col] = df[col].round(decimals)
    return df


def add_prefix(df: pd.DataFrame, prefix: str, exclude: List[str]) -> Tuple[pd.DataFrame, List[str]]:
    excl = set(exclude)
    rename = {c: f'{prefix}{c}' for c in df.columns if c not in excl and not c.startswith(prefix)}
    return df.rename(columns=rename), list(rename.values())


def ensure_quadrant_label(df: pd.DataFrame) -> pd.DataFrame:
    if QUADRANT_COL in df.columns and df[QUADRANT_COL].notna().any():
        return df
    v = pd.to_numeric(df.get('valence'), errors='coerce')
    a = pd.to_numeric(df.get('arousal'), errors='coerce')
    conditions = [(v >= 0) & (a >= 0), (v < 0) & (a >= 0), (v < 0) & (a < 0), (v >= 0) & (a < 0)]
    df = df.copy()
    df[QUADRANT_COL] = np.select(conditions, cfg.QUADRANT_ORDER, default=pd.NA)
    print(f'[INFO] quadrant_label derivado: {df[QUADRANT_COL].value_counts().to_dict()}')
    return df


def make_binary_target(quadrant_series: pd.Series, target_q: str) -> pd.Series:
    return quadrant_series.astype(str).eq(target_q).astype(int)


def feature_origin(feat: str) -> str:
    if feat.startswith('opensmile__'):    return 'openSMILE'
    if feat.startswith('fprank__'):       return 'Fingerprint Rank'
    if feat.startswith('fpband__'):       return 'Fingerprint Band'
    if feat.startswith('rawpeaks_rank__'): return 'Raw Peaks Rank'
    if feat.startswith('rawpeaks_band__'): return 'Raw Peaks Band'
    return 'outros'


ORIGIN_COLORS = {
    'openSMILE':         '#2196F3',
    'Fingerprint Rank':  '#FF9800',
    'Fingerprint Band':  '#4CAF50',
    'Raw Peaks Rank':    '#9C27B0',
    'Raw Peaks Band':    '#E91E63',
    'outros':            '#9E9E9E',
}

SCENARIO_LABELS = {
    '1_opensmile_only':      '1. openSMILE',
    '2_fprank_only':         '2. FP Rank',
    '3_fpband_only':         '3. FP Band',
    '4_allfp_only':          '4. All FP',
    '5_os_fprank':           '5. OS+FPRank',
    '6_os_fpband':           '6. OS+FPBand',
    '7_os_allfp':            '7. OS+AllFP',
    '8_rawpeaks_only':       '8. RawPks',
    '9_os_rawpeaks':         '9. OS+RawPks',
    '10_os_allfp_rawpeaks':  '10. OS+AllFP+Raw',
}
def slabel(s): return SCENARIO_LABELS.get(s, s)

def prefer_norm_magnitude(feat_list: List[str], df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Remove features de magnitude em dB quando existe versão normalizada equivalente.
    Padrões: *_magnitude_db → *_magnitude_norm_block | *_mag_*_db → *_mag_*_norm_block
             energy_db_*   → energy_norm_*           | *_magnitude_db_* → *_magnitude_norm_*
             *_magnitude_mean (sem _norm_) → *_magnitude_mean_norm_block
    """
    df_cols = set(df.columns)
    keep: List[str] = []
    dropped: List[str] = []
    for col in feat_list:
        norm_col = None
        if col.endswith('_magnitude_db'):
            norm_col = col[: -len('_magnitude_db')] + '_magnitude_norm_block'
        elif col.endswith('_db') and '_mag_' in col and '_norm' not in col:
            norm_col = col[: -len('_db')] + '_norm_block'
        elif 'energy_db_' in col:
            norm_col = col.replace('energy_db_', 'energy_norm_')
        elif '_magnitude_db_' in col:
            norm_col = col.replace('_magnitude_db_', '_magnitude_norm_')
        elif '_norm_' not in col and col.endswith('_magnitude_mean'):
            norm_col = col + '_norm_block'
        if norm_col and norm_col in df_cols:
            dropped.append(col)
        else:
            keep.append(col)
    return keep, dropped

## 3. Carregamento dos dados

In [4]:
data_dir = cfg.data_dir()

# ── openSMILE ──────────────────────────────────────────────────────────────
os_df = pd.read_parquet(data_dir / 'comparative_outputs' / 'common' / 'opensmile_blocks.parquet')
os_df = round_times(os_df)
os_df = ensure_quadrant_label(os_df)

os_raw_feat = [c for c in os_df.columns if c.startswith('os_')]
os_rename   = {c: c.replace('os_mean__', 'opensmile__').replace('os_std__', 'opensmile__std__').replace('os_', 'opensmile__') for c in os_raw_feat}
os_df = os_df.rename(columns=os_rename)
os_feat = list(os_rename.values())

print(f'openSMILE: {os_df.shape[0]} blocos | {len(os_feat)} features | {os_df["song_id"].nunique()} músicas')
print('Distribuição quadrantes:')
print(os_df[QUADRANT_COL].value_counts().reindex(cfg.QUADRANT_ORDER).to_string())

openSMILE: 6506 blocos | 520 features | 1802 músicas
Distribuição quadrantes:
quadrant_label
Q1_High_Valence_High_Arousal    3217
Q2_Low_Valence_High_Arousal     1019
Q3_Low_Valence_Low_Arousal      1382
Q4_High_Valence_Low_Arousal      888


In [5]:
# ── Fingerprint Rank (bloco) ───────────────────────────────────────────────
fprank_raw = pd.read_parquet(data_dir / 'fingerprint_rank' / 'fingerprint_rank.parquet')
fprank_raw = round_times(fprank_raw)
fprank_raw, _ = add_prefix(fprank_raw, 'fprank__', cfg.META_COLS)
fprank_feat = numeric_valid_cols(fprank_raw, exclude=cfg.META_COLS)
print(f'FP Rank:  {fprank_raw.shape[0]} blocos | {len(fprank_feat)} features')

# ── Fingerprint Band (bloco) ───────────────────────────────────────────────
fpband_raw = pd.read_parquet(data_dir / 'fingerprint_band_rank' / 'fingerprint_band_rank.parquet')
fpband_raw = round_times(fpband_raw)
fpband_raw, _ = add_prefix(fpband_raw, 'fpband__', cfg.META_COLS)
fpband_feat = numeric_valid_cols(fpband_raw, exclude=cfg.META_COLS)
print(f'FP Band:  {fpband_raw.shape[0]} blocos | {len(fpband_feat)} features')

FP Rank:  6506 blocos | 207 features
FP Band:  6506 blocos | 302 features


## 4. Agregação dos Raw Peaks (longo → bloco)

In [6]:
def aggregate_raw_peaks(path: Path, prefix: str, by_band: bool = False) -> Tuple[Optional[pd.DataFrame], List[str]]:
    if not path.exists():
        print(f'  [SKIP] {path.name}')
        return None, []
    raw = pd.read_parquet(path)
    raw = round_times(raw)
    keys = [k for k in ['song_id', 'block_id', 'block_start_sec', 'block_end_sec'] if k in raw.columns]
    num_cands = ['frequency_hz', 'magnitude_db', 'magnitude_norm', 'magnitude',
                 'midi_note', 'octave', 'semitone', 'peak_rank_global']
    num_cols = [c for c in num_cands if c in raw.columns]
    for c in num_cols:
        raw[c] = pd.to_numeric(raw[c], errors='coerce')

    def _agg(sub_df, pref):
        g = sub_df.groupby(keys)
        agg = g[num_cols].agg(['mean', 'std', 'max'])
        agg.columns = [f'{pref}{col}_{stat}' for col, stat in agg.columns]
        agg[f'{pref}n_peaks'] = g.size()
        return agg.reset_index()

    if by_band and 'band_name' in raw.columns:
        frames = [_agg(raw[raw['band_name'] == b], f'{prefix}{b}_') for b in sorted(raw['band_name'].dropna().unique())]
        if not frames: return None, []
        merged = frames[0]
        for pf in frames[1:]:
            merged = merged.merge(pf, on=keys, how='outer')
    else:
        merged = _agg(raw, prefix)

    feat_cols = [c for c in merged.columns if c.startswith(prefix)]
    print(f'  {prefix}: {merged.shape[0]} blocos | {len(feat_cols)} features')
    return merged, feat_cols


print('Agregando raw peaks...')
rr_df, rr_feat = aggregate_raw_peaks(data_dir / 'fingerprint_rank' / 'fingerprint_rank_raw_peaks.parquet',
                                     prefix='rawpeaks_rank__', by_band=False)
rb_df, rb_feat = aggregate_raw_peaks(data_dir / 'fingerprint_band_rank' / 'fingerprint_band_rank_raw_peaks.parquet',
                                     prefix='rawpeaks_band__', by_band=True)

HAS_RR = cfg.use_raw_peaks and rr_df is not None
HAS_RB = cfg.use_raw_peaks and rb_df is not None
HAS_RAW = HAS_RR or HAS_RB
print(f'Raw peaks: rank={HAS_RR}, band={HAS_RB}')

Agregando raw peaks...
  rawpeaks_rank__: 6506 blocos | 19 features
  rawpeaks_band__: 6506 blocos | 100 features
Raw peaks: rank=True, band=True


## 5. Montagem do dataset integrado e cenários

In [7]:
def left_join(base: pd.DataFrame, right: pd.DataFrame, feat_cols: List[str], name: str) -> Tuple[pd.DataFrame, List[str]]:
    keys_ok = [k for k in JOIN_KEYS if k in right.columns]
    right_sel = right[keys_ok + feat_cols].drop_duplicates(subset=keys_ok)
    merged = base.merge(right_sel, on=keys_ok, how='left')
    added = [f for f in feat_cols if f in merged.columns]
    cov = merged[added[0]].notna().mean() if added else 0.0
    print(f'  [{name}] {merged.shape[0]} blocos | {len(added)} features | cov {cov:.1%}')
    return merged, added


base_meta = [c for c in ['song_id','filename','block_id','block_start_sec','block_end_sec',
                          'block_duration_sec','valence','arousal','quadrant', QUADRANT_COL] if c in os_df.columns]
dataset = os_df[base_meta + os_feat].copy()

dataset, fprank_feat = left_join(dataset, fprank_raw, fprank_feat, 'FP Rank')
dataset, fpband_feat = left_join(dataset, fpband_raw, fpband_feat, 'FP Band')
if HAS_RR: dataset, rr_feat = left_join(dataset, rr_df, rr_feat, 'Raw Rank')
if HAS_RB: dataset, rb_feat = left_join(dataset, rb_df, rb_feat, 'Raw Band')

all_raw_v = list(dict.fromkeys((rr_feat if HAS_RR else []) + (rb_feat if HAS_RB else [])))
print(f'\nDataset: {dataset.shape} | músicas: {dataset["song_id"].nunique()}')

  [FP Rank] 6506 blocos | 207 features | cov 100.0%
  [FP Band] 6506 blocos | 302 features | cov 100.0%
  [Raw Rank] 6506 blocos | 19 features | cov 99.5%
  [Raw Band] 6506 blocos | 100 features | cov 100.0%

Dataset: (6506, 1158) | músicas: 1802


In [8]:
all_meta = set(cfg.META_COLS + ['block_id', 'block_duration_sec', 'filename', 'quadrant'])

def vf(cols): return [c for c in numeric_valid_cols(dataset, exclude=list(all_meta)) if c in cols]

os_fv    = vf(os_feat)
fpr_fv   = vf(fprank_feat)
fpb_fv   = vf(fpband_feat)
_rr_fv   = vf(rr_feat) if HAS_RR else []
_rb_fv   = vf(rb_feat) if HAS_RB else []

# Priorizar magnitude normalizada: remover versões em dB com par normalizado
fpr_fv,  _d1 = prefer_norm_magnitude(fpr_fv, dataset)
fpb_fv,  _d2 = prefer_norm_magnitude(fpb_fv, dataset)
_rr_fv, _d3  = prefer_norm_magnitude(_rr_fv, dataset)
_rb_fv, _d4  = prefer_norm_magnitude(_rb_fv, dataset)
_all_dropped = _d1 + _d2 + _d3 + _d4
print(f'Features dB removidas (par norm disponivel): {len(_all_dropped)}')
for _d in sorted(_all_dropped)[:20]: print(f'  - {_d}')
if len(_all_dropped) > 20: print(f'  ... e mais {len(_all_dropped) - 20}')
print()

allfp_fv = list(dict.fromkeys(fpr_fv + fpb_fv))
raw_fv   = list(dict.fromkeys(_rr_fv + _rb_fv))

SCENARIOS: Dict[str, List[str]] = {
    '1_opensmile_only': os_fv,
    '2_fprank_only':    fpr_fv,
    '3_fpband_only':    fpb_fv,
    '4_allfp_only':     allfp_fv,
    '5_os_fprank':      list(dict.fromkeys(os_fv + fpr_fv)),
    '6_os_fpband':      list(dict.fromkeys(os_fv + fpb_fv)),
    '7_os_allfp':       list(dict.fromkeys(os_fv + allfp_fv)),
}
if HAS_RAW and raw_fv:
    SCENARIOS['8_rawpeaks_only']      = raw_fv
    SCENARIOS['9_os_rawpeaks']        = list(dict.fromkeys(os_fv + raw_fv))
    SCENARIOS['10_os_allfp_rawpeaks'] = list(dict.fromkeys(os_fv + allfp_fv + raw_fv))

print('Cenários:')
for k, v in SCENARIOS.items():
    print(f'  {k}: {len(v)} features')

Features dB removidas (par norm disponivel): 97
  - fpband__energy_db_high
  - fpband__energy_db_low
  - fpband__energy_db_mid
  - fpband__energy_db_very_high
  - fpband__fp_high_mag_mean_db
  - fpband__fp_high_mag_std_db
  - fpband__fp_high_top_10_magnitude_db
  - fpband__fp_high_top_1_magnitude_db
  - fpband__fp_high_top_2_magnitude_db
  - fpband__fp_high_top_3_magnitude_db
  - fpband__fp_high_top_4_magnitude_db
  - fpband__fp_high_top_5_magnitude_db
  - fpband__fp_high_top_6_magnitude_db
  - fpband__fp_high_top_7_magnitude_db
  - fpband__fp_high_top_8_magnitude_db
  - fpband__fp_high_top_9_magnitude_db
  - fpband__fp_low_mag_mean_db
  - fpband__fp_low_mag_std_db
  - fpband__fp_low_top_10_magnitude_db
  - fpband__fp_low_top_1_magnitude_db
  ... e mais 77

Cenários:
  1_opensmile_only: 520 features
  2_fprank_only: 176 features
  3_fpband_only: 248 features
  4_allfp_only: 424 features
  5_os_fprank: 696 features
  6_os_fpband: 768 features
  7_os_allfp: 944 features
  8_rawpeaks_only

In [9]:
# balanço binário por quadrante
q_labels = dataset[QUADRANT_COL].astype(str)
balance_rows = []
for q in cfg.QUADRANT_ORDER:
    n_pos = int((q_labels == q).sum())
    n_neg = len(q_labels) - n_pos
    balance_rows.append({'quadrant': q, 'short': cfg.QUADRANT_SHORT[q],
                         'n_positive': n_pos, 'n_negative': n_neg,
                         'positive_rate': n_pos / len(q_labels)})
class_balance = pd.DataFrame(balance_rows)
display(Markdown('### Balanço binário por quadrante'))
display(class_balance)

### Balanço binário por quadrante

,quadrant,short,n_positive,n_negative,positive_rate
0,Q1_High_Valence_High_Arousal,Q1,3217,3289,0.494467
1,Q2_Low_Valence_High_Arousal,Q2,1019,5487,0.156625
2,Q3_Low_Valence_Low_Arousal,Q3,1382,5124,0.212419
3,Q4_High_Valence_Low_Arousal,Q4,888,5618,0.136489


## 6. Modelos, pipeline e métricas

In [10]:
def build_models(cfg: Config) -> Dict[str, object]:
    models = {
        'RidgeClassifier':    RidgeClassifier(class_weight='balanced', random_state=cfg.random_state),
        'LogisticRegression': LogisticRegression(max_iter=3000, class_weight='balanced',
                                                  random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        'LinearSVM':          CalibratedClassifierCV(LinearSVC(class_weight='balanced',
                                                                random_state=cfg.random_state, max_iter=2000)),
        'RandomForest':       RandomForestClassifier(n_estimators=cfg.n_estimators, class_weight='balanced_subsample',
                                                      random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        'ExtraTrees':         ExtraTreesClassifier(n_estimators=cfg.n_estimators, class_weight='balanced',
                                                    random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        'GradientBoosting':   GradientBoostingClassifier(n_estimators=cfg.n_estimators,
                                                          random_state=cfg.random_state),
    }
    if HAS_HGBC:
        models['HistGradientBoosting'] = HistGradientBoostingClassifier(random_state=cfg.random_state)
    return models


NEEDS_SCALE = {'RidgeClassifier', 'LogisticRegression', 'LinearSVM'}


def build_pipeline(model, n_feats: int, selector_k: Optional[int] = None, scale: bool = True) -> Pipeline:
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if selector_k is not None:
        k_eff = min(selector_k, n_feats)
        steps.append(('selector', SelectKBest(score_func=f_classif, k=k_eff)))
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)


def get_positive_scores(pipeline, X_va) -> Optional[np.ndarray]:
    """Retorna scores de probabilidade ou decision_function para a classe positiva."""
    est = pipeline.named_steps['model']
    if hasattr(est, 'predict_proba'):
        p = est.predict_proba(X_va if not hasattr(pipeline[:-1], 'transform') else pipeline[:-1].transform(X_va))
        return p[:, 1] if p.shape[1] == 2 else None
    if hasattr(est, 'decision_function'):
        X_t = pipeline[:-1].transform(X_va) if len(pipeline) > 1 else X_va
        raw = est.decision_function(X_t)
        if raw.ndim > 1: raw = raw[:, -1]
        return 1.0 / (1.0 + np.exp(-np.clip(raw, -60, 60)))
    return None


def binary_metrics(y_true, y_pred, scores=None) -> Dict[str, float]:
    m = {
        'positive_f1':        f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'positive_precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'positive_recall':    recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'balanced_accuracy':  balanced_accuracy_score(y_true, y_pred),
        'accuracy':           accuracy_score(y_true, y_pred),
    }
    if scores is not None and len(np.unique(y_true)) == 2:
        try:
            m['roc_auc']           = roc_auc_score(y_true, scores)
            m['average_precision'] = average_precision_score(y_true, scores)
        except Exception:
            m['roc_auc'] = m['average_precision'] = np.nan
    else:
        m['roc_auc'] = m['average_precision'] = np.nan
    return m


def get_importance(pipeline) -> Optional[np.ndarray]:
    est = pipeline.named_steps.get('model')
    if isinstance(est, CalibratedClassifierCV):
        est = getattr(est, 'calibrated_classifiers_', [None])
        est = est[0].estimator if est else None
    if est is None: return None
    if hasattr(est, 'feature_importances_'): return est.feature_importances_
    if hasattr(est, 'coef_'):
        coef = est.coef_
        return np.abs(coef).mean(axis=0) if coef.ndim > 1 else np.abs(coef)
    return None


BINARY_METRICS = ['positive_f1', 'positive_precision', 'positive_recall',
                  'balanced_accuracy', 'accuracy', 'roc_auc', 'average_precision']

## 7. Loop principal de avaliação GroupKFold

In [11]:
gkf    = GroupKFold(n_splits=cfg.n_splits)
models = build_models(cfg)
groups = dataset['song_id'].astype(str).values
q_series = dataset[QUADRANT_COL].astype(str)

fold_records       = []
importance_records = []
# OOF binary predictions: {scenario: {quadrant: {model: {selector: np.array}}}
oof_store: Dict = {}

# total iterations for progress bar
total_iter = len(SCENARIOS) * len(cfg.QUADRANT_ORDER) * len(models) * (1 + len(cfg.selector_k_values))
pbar = tqdm(total=total_iter, desc='Avaliação binária')

for scenario_name, feat_cols in SCENARIOS.items():
    feat_cols = [f for f in feat_cols if f in dataset.columns]
    if not feat_cols:
        pbar.update(len(cfg.QUADRANT_ORDER) * len(models) * (1 + len(cfg.selector_k_values)))
        continue

    X_all = dataset[feat_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    oof_store[scenario_name] = {}

    for target_q in cfg.QUADRANT_ORDER:
        y_bin = make_binary_target(q_series, target_q)
        valid_mask = y_bin.notna() & X_all.notna().any(axis=1)
        X_v = X_all[valid_mask]
        y_v = y_bin[valid_mask]
        g_v = groups[valid_mask]

        # pre-compute fold indices (same for all models/selectors in this scenario×quadrant)
        fold_splits = list(gkf.split(X_v, y_v, g_v))

        oof_store[scenario_name][target_q] = {}

        selector_options = [None] + [k for k in cfg.selector_k_values if k < len(feat_cols)]

        for model_name, base_model in models.items():
            scale = model_name in NEEDS_SCALE
            oof_store[scenario_name][target_q][model_name] = {}

            for sel_k in selector_options:
                sel_name = 'none' if sel_k is None else f'SelectKBest(k={sel_k})'
                oof_pred  = np.full(len(y_v), fill_value=-1, dtype=int)
                oof_score = np.full(len(y_v), fill_value=np.nan)
                fold_imps = []

                for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
                    pipe = build_pipeline(clone(base_model), len(feat_cols), sel_k, scale)
                    X_tr, X_va = X_v.iloc[tr_idx].values, X_v.iloc[va_idx].values
                    y_tr, y_va = y_v.iloc[tr_idx].values, y_v.iloc[va_idx].values

                    try:
                        pipe.fit(X_tr, y_tr)
                        pred   = pipe.predict(X_va).astype(int)
                        scores = get_positive_scores(pipe, X_va)
                        oof_pred[va_idx]  = pred
                        if scores is not None:
                            oof_score[va_idx] = scores
                        mets   = binary_metrics(y_va, pred, scores)
                        status = 'ok'

                        # feature importance: map back through selector if present
                        sel_step = pipe.named_steps.get('selector')
                        imp_raw  = get_importance(pipe)
                        if imp_raw is not None:
                            if sel_step is not None:
                                # expand back to full feature space
                                mask_sel = sel_step.get_support()
                                imp_full = np.zeros(len(feat_cols))
                                imp_full[mask_sel] = imp_raw
                            else:
                                imp_full = imp_raw if len(imp_raw) == len(feat_cols) else None
                            if imp_full is not None:
                                fold_imps.append(imp_full)

                    except Exception as e:
                        mets   = {k: np.nan for k in BINARY_METRICS}
                        status = f'error:{e}'

                    fold_records.append({
                        'scenario': scenario_name, 'quadrant': target_q,
                        'short': cfg.QUADRANT_SHORT[target_q],
                        'model': model_name, 'selector': sel_name,
                        'fold': fold_idx, 'n_features': len(feat_cols),
                        'n_train': len(tr_idx), 'n_val': len(va_idx),
                        'status': status, **mets,
                    })

                oof_store[scenario_name][target_q][model_name][sel_name] = (oof_pred, oof_score)

                if fold_imps:
                    mean_imp = np.mean(fold_imps, axis=0)
                    for feat, val in zip(feat_cols, mean_imp):
                        importance_records.append({
                            'scenario': scenario_name, 'quadrant': target_q,
                            'model': model_name, 'selector': sel_name,
                            'feature': feat, 'importance': val,
                        })

                pbar.update(1)

pbar.close()
fold_df = pd.DataFrame(fold_records)
print(f'\nRegistros de folds: {len(fold_df)}')

Avaliação binária:   0%|          | 0/1400 [00:00<?, ?it/s]

  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



Registros de folds: 6580


## 8. Agregação e melhor modelo por cenário × quadrante

In [12]:
ok_df = fold_df[fold_df['status'] == 'ok'].copy()

agg_dict = {}
for m in BINARY_METRICS:
    if m in ok_df.columns:
        agg_dict[f'{m}_mean'] = (m, 'mean')
        agg_dict[f'{m}_std']  = (m, 'std')
agg_dict['n_folds'] = ('fold', 'count')

agg_df = (
    ok_df.groupby(['scenario', 'quadrant', 'short', 'model', 'selector', 'n_features'])
    .agg(**agg_dict)
    .reset_index()
    .sort_values(['quadrant', 'scenario', 'positive_f1_mean'], ascending=[True, True, False])
)

display(Markdown('### Resultados agregados (top 20)'))
display(agg_df[['scenario','quadrant','model','selector','positive_f1_mean','positive_f1_std',
                'balanced_accuracy_mean','roc_auc_mean']].head(20))

### Resultados agregados (top 20)

,scenario,quadrant,model,selector,positive_f1_mean,positive_f1_std,balanced_accuracy_mean,roc_auc_mean
9,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,GradientBoosting,none,0.754691,0.021064,0.754866,0.829171
11,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,HistGradientBoosting,SelectKBest(k=200),0.754586,0.024141,0.754369,0.827789
6,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,GradientBoosting,SelectKBest(k=200),0.753594,0.020953,0.753184,0.826608
1,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,ExtraTrees,SelectKBest(k=200),0.750258,0.020017,0.752719,0.823511
14,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,HistGradientBoosting,none,0.749726,0.015505,0.751063,0.830896
10,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,HistGradientBoosting,SelectKBest(k=100),0.749114,0.028116,0.749372,0.820693
26,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,RandomForest,SelectKBest(k=200),0.748840,0.021967,0.754171,0.826198
4,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,ExtraTrees,none,0.745807,0.019378,0.746760,0.821670
3,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,ExtraTrees,SelectKBest(k=60),0.745436,0.032190,0.746387,0.818281
29,10_os_allfp_rawpeaks,Q1_High_Valence_High_Arousal,RandomForest,none,0.745224,0.022430,0.751038,0.826948


In [13]:
# melhor modelo por cenário × quadrante
best_by_scen_q = (
    agg_df.sort_values('positive_f1_mean', ascending=False)
    .groupby(['scenario', 'quadrant'])
    .first()
    .reset_index()
    .rename(columns={'model': 'best_model', 'selector': 'best_selector'})
)

# melhor cenário por quadrante
best_by_q = (
    best_by_scen_q.sort_values('positive_f1_mean', ascending=False)
    .groupby('quadrant')
    .first()
    .reset_index()
)
display(Markdown('### Melhor cenário por quadrante'))
display(best_by_q[['quadrant','scenario','best_model','best_selector',
                    'positive_f1_mean','positive_f1_std','balanced_accuracy_mean','roc_auc_mean']])

### Melhor cenário por quadrante

,quadrant,scenario,best_model,best_selector,positive_f1_mean,positive_f1_std,balanced_accuracy_mean,roc_auc_mean
0,Q1_High_Valence_High_Arousal,5_os_fprank,ExtraTrees,SelectKBest(k=200),0.757723,0.022705,0.760438,0.824019
1,Q2_Low_Valence_High_Arousal,10_os_allfp_rawpeaks,LogisticRegression,SelectKBest(k=100),0.400608,0.015173,0.673293,0.721021
2,Q3_Low_Valence_Low_Arousal,7_os_allfp,RidgeClassifier,SelectKBest(k=200),0.599369,0.040631,0.782178,0.857662
3,Q4_High_Valence_Low_Arousal,9_os_rawpeaks,LogisticRegression,SelectKBest(k=60),0.360636,0.060658,0.678204,0.738121


## 9. Comparação: fusão vs openSMILE isolado

In [14]:
def classify_gain_binary(row) -> str:
    df1  = row.get('delta_positive_f1', 0.0)
    dba  = row.get('delta_balanced_accuracy', 0.0)
    drec = row.get('delta_positive_recall', 0.0)
    if df1 > 0.015 and dba > 0.01:
        return 'melhora consistente'
    elif df1 > 0.01:
        return 'melhora em F1 positivo'
    elif drec > 0.02:
        return 'melhora em recall positivo'
    elif df1 > 0.003 or dba > 0.003:
        return 'melhora pequena'
    return 'sem melhora relevante'


os_ref = best_by_scen_q[best_by_scen_q['scenario'] == '1_opensmile_only'][
    ['quadrant', 'positive_f1_mean', 'positive_precision_mean', 'positive_recall_mean',
     'balanced_accuracy_mean', 'roc_auc_mean', 'average_precision_mean']
].rename(columns=lambda c: c.replace('_mean', '_os') if '_mean' in c else c)

fusion_scens = [s for s in best_by_scen_q['scenario'].unique() if s != '1_opensmile_only']
delta_df = best_by_scen_q[best_by_scen_q['scenario'].isin(fusion_scens)].copy()
delta_df = delta_df.merge(os_ref, on='quadrant', how='left')

for m in ['positive_f1', 'positive_precision', 'positive_recall', 'balanced_accuracy', 'roc_auc']:
    delta_df[f'delta_{m}'] = delta_df[f'{m}_mean'] - delta_df[f'{m}_os']

delta_df['delta_positive_f1_pct'] = 100 * delta_df['delta_positive_f1'] / delta_df['positive_f1_os'].replace(0, np.nan)
delta_df['ganho'] = delta_df.apply(classify_gain_binary, axis=1)

display(Markdown('### Delta vs openSMILE isolado (por quadrante)'))
display(delta_df[['quadrant','scenario','best_model','positive_f1_mean','positive_f1_os',
                   'delta_positive_f1','delta_positive_f1_pct','delta_balanced_accuracy','ganho']]
        .sort_values(['quadrant','delta_positive_f1'], ascending=[True, False]))

### Delta vs openSMILE isolado (por quadrante)

,quadrant,scenario,best_model,positive_f1_mean,positive_f1_os,delta_positive_f1,delta_positive_f1_pct,delta_balanced_accuracy,ganho
16,Q1_High_Valence_High_Arousal,5_os_fprank,ExtraTrees,0.757723,0.756882,0.000841,0.111155,0.001465,sem melhora relevante
32,Q1_High_Valence_High_Arousal,9_os_rawpeaks,HistGradientBoosting,0.757042,0.756882,0.000160,0.021131,-0.000096,sem melhora relevante
20,Q1_High_Valence_High_Arousal,6_os_fpband,GradientBoosting,0.755495,0.756882,-0.001387,-0.183281,-0.004537,sem melhora relevante
0,Q1_High_Valence_High_Arousal,10_os_allfp_rawpeaks,GradientBoosting,0.754691,0.756882,-0.002191,-0.289417,-0.004107,sem melhora relevante
24,Q1_High_Valence_High_Arousal,7_os_allfp,HistGradientBoosting,0.752655,0.756882,-0.004227,-0.558428,-0.006223,sem melhora relevante
12,Q1_High_Valence_High_Arousal,4_allfp_only,GradientBoosting,0.731598,0.756882,-0.025284,-3.340558,-0.026387,sem melhora relevante
4,Q1_High_Valence_High_Arousal,2_fprank_only,LogisticRegression,0.720742,0.756882,-0.036140,-4.774825,-0.043083,sem melhora relevante
8,Q1_High_Valence_High_Arousal,3_fpband_only,ExtraTrees,0.705794,0.756882,-0.051088,-6.749794,-0.063640,sem melhora relevante
28,Q1_High_Valence_High_Arousal,8_rawpeaks_only,ExtraTrees,0.693237,0.756882,-0.063645,-8.408819,-0.067692,sem melhora relevante
1,Q2_Low_Valence_High_Arousal,10_os_allfp_rawpeaks,LogisticRegression,0.400608,0.400096,0.000512,0.127863,-0.000843,sem melhora relevante


In [15]:
# precision vs recall por quadrante
pr_rows = []
for q in cfg.QUADRANT_ORDER:
    sub = agg_df[(agg_df['quadrant'] == q) & (~agg_df['model'].str.startswith('Dummy'))]
    if sub.empty: continue
    for _, row in sub.iterrows():
        pr_rows.append({'quadrant': q, 'short': cfg.QUADRANT_SHORT[q],
                        'scenario': row['scenario'], 'model': row['model'],
                        'positive_precision_mean': row.get('positive_precision_mean', np.nan),
                        'positive_recall_mean': row.get('positive_recall_mean', np.nan),
                        'positive_f1_mean': row.get('positive_f1_mean', np.nan)})
pr_df = pd.DataFrame(pr_rows)

## 10. Matrizes de confusão binárias (OOF)

In [16]:
def get_best_oof(scenario: str, quadrant: str) -> Tuple[str, str, np.ndarray, np.ndarray, np.ndarray]:
    """Retorna (model, selector, y_true, oof_pred, oof_score) para o melhor modelo/selector."""
    sub = agg_df[(agg_df['scenario'] == scenario) & (agg_df['quadrant'] == quadrant)]
    if sub.empty:
        return None, None, None, None, None
    best = sub.sort_values('positive_f1_mean', ascending=False).iloc[0]
    model_name = best['model']
    sel_name   = best['selector']
    oof_data   = oof_store.get(scenario, {}).get(quadrant, {}).get(model_name, {}).get(sel_name)
    if oof_data is None:
        return model_name, sel_name, None, None, None
    oof_pred, oof_score = oof_data
    y_bin = make_binary_target(q_series, quadrant).values
    valid = oof_pred >= 0
    return model_name, sel_name, y_bin[valid], oof_pred[valid], oof_score[valid]


cm_records = []
for q in cfg.QUADRANT_ORDER:
    short = cfg.QUADRANT_SHORT[q]
    # best FP-only scenario for this quadrant
    fp_scens = best_by_scen_q[(best_by_scen_q['quadrant'] == q) &
                               (best_by_scen_q['scenario'].isin(['2_fprank_only','3_fpband_only','4_allfp_only']))]
    best_fp_scen = fp_scens.sort_values('positive_f1_mean', ascending=False).iloc[0]['scenario'] if not fp_scens.empty else '2_fprank_only'
    # best fusion scenario for this quadrant
    best_fus_scen = delta_df[delta_df['quadrant'] == q].sort_values('delta_positive_f1', ascending=False)
    best_fus_scen = best_fus_scen.iloc[0]['scenario'] if not best_fus_scen.empty else '7_os_allfp'

    for scen_name, label in [('1_opensmile_only', 'openSMILE'), (best_fp_scen, 'Best FP'), (best_fus_scen, 'Best Fusion')]:
        m, sel, y_t, y_p, _ = get_best_oof(scen_name, q)
        if y_t is None: continue
        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        cm_records.append({
            'quadrant': q, 'short': short, 'scenario_label': label, 'scenario': scen_name, 'model': m,
            'TN': int(cm[0,0]), 'FP': int(cm[0,1]), 'FN': int(cm[1,0]), 'TP': int(cm[1,1]),
        })

cm_df = pd.DataFrame(cm_records)
display(Markdown('### Matrizes de confusão binárias (OOF)'))
display(cm_df)

### Matrizes de confusão binárias (OOF)

,quadrant,short,scenario_label,scenario,model,TN,FP,FN,TP
0,Q1_High_Valence_High_Arousal,Q1,openSMILE,1_opensmile_only,GradientBoosting,2484,805,769,2448
1,Q1_High_Valence_High_Arousal,Q1,Best FP,4_allfp_only,GradientBoosting,2382,907,839,2378
2,Q1_High_Valence_High_Arousal,Q1,Best Fusion,5_os_fprank,ExtraTrees,2499,790,775,2442
3,Q2_Low_Valence_High_Arousal,Q2,openSMILE,1_opensmile_only,RidgeClassifier,3962,1525,381,638
4,Q2_Low_Valence_High_Arousal,Q2,Best FP,4_allfp_only,LogisticRegression,3875,1612,429,590
5,Q2_Low_Valence_High_Arousal,Q2,Best Fusion,10_os_allfp_rawpeaks,LogisticRegression,4002,1485,391,628
6,Q3_Low_Valence_Low_Arousal,Q3,openSMILE,1_opensmile_only,LogisticRegression,3925,1199,297,1085
7,Q3_Low_Valence_Low_Arousal,Q3,Best FP,4_allfp_only,RidgeClassifier,3824,1300,341,1041
8,Q3_Low_Valence_Low_Arousal,Q3,Best Fusion,7_os_allfp,RidgeClassifier,3978,1146,293,1089
9,Q4_High_Valence_Low_Arousal,Q4,openSMILE,1_opensmile_only,LogisticRegression,3576,2042,265,623


## 11. Importância das features por origem

In [17]:
importance_df = pd.DataFrame(importance_records)
imp_by_origin = pd.DataFrame()
top20_by_q: Dict[str, pd.DataFrame] = {}

if not importance_df.empty:
    importance_df['origin'] = importance_df['feature'].map(feature_origin)

    # importance by origin per scenario × quadrant
    imp_by_origin = (
        importance_df.groupby(['scenario','quadrant','model','origin'])['importance']
        .sum().reset_index()
    )
    total_imp = imp_by_origin.groupby(['scenario','quadrant','model'])['importance'].transform('sum')
    imp_by_origin['importance_pct'] = 100 * imp_by_origin['importance'] / total_imp.replace(0, np.nan)

    # top 20 features per quadrant (scenario 7, tree models)
    tree_models = ['ExtraTrees','RandomForest','GradientBoosting','HistGradientBoosting']
    for q in cfg.QUADRANT_ORDER:
        sub = importance_df[
            (importance_df['quadrant'] == q) &
            (importance_df['scenario'] == '7_os_allfp') &
            (importance_df['model'].isin(tree_models)) &
            (importance_df['selector'] == 'none')
        ]
        if sub.empty:
            sub = importance_df[(importance_df['quadrant'] == q) & (importance_df['scenario'] == '7_os_allfp')]
        if not sub.empty:
            top20 = (
                sub.groupby('feature')['importance'].mean()
                .sort_values(ascending=False).head(20).reset_index()
            )
            top20['origin'] = top20['feature'].map(feature_origin)
            top20_by_q[q] = top20

    display(Markdown('### Importância por Origem — cenário 7, Q1 (exemplo)'))
    sub7q1 = imp_by_origin[(imp_by_origin['scenario']=='7_os_allfp') &
                            (imp_by_origin['quadrant']=='Q1_High_Valence_High_Arousal')]
    display(sub7q1.sort_values('importance_pct', ascending=False).head(15))
else:
    print('[INFO] Nenhuma importância de feature registrada')

### Importância por Origem — cenário 7, Q1 (exemplo)

,scenario,quadrant,model,origin,importance,importance_pct
341,7_os_allfp,Q1_High_Valence_High_Arousal,GradientBoosting,openSMILE,3.410253,68.205053
350,7_os_allfp,Q1_High_Valence_High_Arousal,RandomForest,openSMILE,3.234763,64.695252
338,7_os_allfp,Q1_High_Valence_High_Arousal,ExtraTrees,openSMILE,3.200560,64.011198
347,7_os_allfp,Q1_High_Valence_High_Arousal,LogisticRegression,openSMILE,220.405039,59.952802
353,7_os_allfp,Q1_High_Valence_High_Arousal,RidgeClassifier,openSMILE,102.774113,55.897794
344,7_os_allfp,Q1_High_Valence_High_Arousal,LinearSVM,openSMILE,160.190024,53.790123
342,7_os_allfp,Q1_High_Valence_High_Arousal,LinearSVM,Fingerprint Band,76.889707,25.818754
351,7_os_allfp,Q1_High_Valence_High_Arousal,RidgeClassifier,Fingerprint Band,46.475212,25.277395
348,7_os_allfp,Q1_High_Valence_High_Arousal,RandomForest,Fingerprint Band,1.190913,23.818251
345,7_os_allfp,Q1_High_Valence_High_Arousal,LogisticRegression,Fingerprint Band,85.488358,23.253854


## 12. Gráficos

In [18]:
PLOTLY_TEMPLATE = 'plotly_white'

SCENARIO_TYPE = {
    '1_opensmile_only':     'openSMILE',
    '2_fprank_only':        'Apenas FP',
    '3_fpband_only':        'Apenas FP',
    '4_allfp_only':         'Apenas FP',
    '5_os_fprank':          'Fusao OS+FP',
    '6_os_fpband':          'Fusao OS+FP',
    '7_os_allfp':           'Fusao OS+FP',
    '8_rawpeaks_only':      'Apenas FP',
    '9_os_rawpeaks':        'Fusao OS+FP',
    '10_os_allfp_rawpeaks': 'Fusao OS+FP',
}

SCENARIO_TYPE_COLORS = {
    'openSMILE':    '#2196F3',
    'Apenas FP':    '#FF9800',
    'Fusao OS+FP':  '#4CAF50',
}


def stype(s: str) -> str:
    return SCENARIO_TYPE.get(s, 'outros')


def save_plot(fig, name: str, cfg) -> None:
    fig.update_layout(template=PLOTLY_TEMPLATE)
    if cfg.export_html:
        path = cfg.figures_dir() / f'{name}.html'
        fig.write_html(str(path), include_plotlyjs='cdn')
        print(f'Salvo: {path.name}')
    if cfg.show_figures:
        fig.show()

In [19]:
# Grafico 1: Positive F1 por cenario x quadrante
plot_f1 = best_by_scen_q.copy()
plot_f1['label'] = plot_f1['scenario'].map(slabel)
plot_f1['tipo']  = plot_f1['scenario'].map(stype)
plot_f1['short'] = plot_f1['quadrant'].map(cfg.QUADRANT_SHORT)

label_order_f1 = (
    plot_f1.groupby('label')['positive_f1_mean'].mean()
    .sort_values(ascending=True).index.tolist()
)

fig_f1 = px.bar(
    plot_f1.sort_values('scenario'),
    x='positive_f1_mean', y='label',
    color='tipo', facet_col='short',
    orientation='h',
    error_x='positive_f1_std',
    text=plot_f1.sort_values('scenario')['positive_f1_mean'].map('{:.3f}'.format),
    hover_data={'best_model': True, 'n_features': True,
                'balanced_accuracy_mean': ':.3f', 'roc_auc_mean': ':.3f',
                'positive_f1_std': ':.4f', 'tipo': False},
    color_discrete_map=SCENARIO_TYPE_COLORS,
    category_orders={'label': label_order_f1},
    title='Positive F1 por Cenario e Quadrante (GroupKFold n=5)',
    labels={'positive_f1_mean': 'F1 positivo', 'label': '', 'tipo': 'Tipo'},
)
fig_f1.update_traces(textposition='outside')
fig_f1.update_layout(
    height=480, legend_title='Tipo de Cenario',
    margin=dict(l=10, r=40, t=60, b=10),
)
fig_f1.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig_f1.update_xaxes(matches=None)

save_plot(fig_f1, 'fig_positive_f1_por_cenario_quadrante', cfg)

Salvo: fig_positive_f1_por_cenario_quadrante.html


In [20]:
# Grafico 2: Delta Positive F1 vs openSMILE
if not delta_df.empty:
    plot_delta = delta_df.copy()
    plot_delta['label'] = plot_delta['scenario'].map(slabel)
    plot_delta['short'] = plot_delta['quadrant'].map(cfg.QUADRANT_SHORT)
    plot_delta['cor'] = plot_delta['delta_positive_f1'].apply(
        lambda d: 'Melhorou' if d > 0 else 'Piorou'
    )

    label_order_d = (
        plot_delta.groupby('label')['delta_positive_f1'].mean()
        .sort_values(ascending=False).index.tolist()
    )

    fig_delta = px.bar(
        plot_delta.sort_values('scenario'),
        x='delta_positive_f1', y='label',
        color='cor', facet_col='short',
        orientation='h',
        text=plot_delta.sort_values('scenario')['delta_positive_f1'].map('{:+.3f}'.format),
        hover_data={'best_model': True, 'positive_f1_mean': ':.4f',
                    'positive_f1_os': ':.4f', 'delta_positive_f1_pct': ':.2f',
                    'delta_balanced_accuracy': ':.4f', 'ganho': True, 'cor': False},
        color_discrete_map={'Melhorou': '#4CAF50', 'Piorou': '#F44336'},
        category_orders={'label': label_order_d},
        title='Delta Positive F1 — Fusao vs openSMILE Isolado (positivo = melhor)',
        labels={'delta_positive_f1': 'delta F1 (fusao - openSMILE)', 'label': '', 'cor': 'Impacto'},
    )
    fig_delta.update_traces(textposition='outside')
    fig_delta.add_vline(x=0, line_dash='dash', line_color='black', line_width=1.5)
    fig_delta.update_layout(
        height=480, legend_title='Impacto',
        margin=dict(l=10, r=40, t=60, b=10),
    )
    fig_delta.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig_delta.update_xaxes(matches=None)

    save_plot(fig_delta, 'fig_delta_positive_f1', cfg)

Salvo: fig_delta_positive_f1.html


In [21]:
# Grafico 3: Balanced Accuracy por cenario x quadrante
plot_ba = best_by_scen_q.copy()
plot_ba['label'] = plot_ba['scenario'].map(slabel)
plot_ba['tipo']  = plot_ba['scenario'].map(stype)
plot_ba['short'] = plot_ba['quadrant'].map(cfg.QUADRANT_SHORT)

label_order_ba = (
    plot_ba.groupby('label')['balanced_accuracy_mean'].mean()
    .sort_values(ascending=True).index.tolist()
)

fig_ba = px.bar(
    plot_ba.sort_values('scenario'),
    x='balanced_accuracy_mean', y='label',
    color='tipo', facet_col='short',
    orientation='h',
    error_x='balanced_accuracy_std' if 'balanced_accuracy_std' in plot_ba.columns else None,
    text=plot_ba.sort_values('scenario')['balanced_accuracy_mean'].map('{:.3f}'.format),
    hover_data={'best_model': True, 'n_features': True,
                'positive_f1_mean': ':.3f', 'tipo': False},
    color_discrete_map=SCENARIO_TYPE_COLORS,
    category_orders={'label': label_order_ba},
    title='Balanced Accuracy por Cenario e Quadrante',
    labels={'balanced_accuracy_mean': 'Balanced Accuracy', 'label': '', 'tipo': 'Tipo'},
)
fig_ba.update_traces(textposition='outside')
fig_ba.update_layout(
    height=480, legend_title='Tipo de Cenario',
    margin=dict(l=10, r=40, t=60, b=10),
)
fig_ba.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig_ba.update_xaxes(matches=None)

save_plot(fig_ba, 'fig_balanced_accuracy_por_quadrante', cfg)

Salvo: fig_balanced_accuracy_por_quadrante.html


In [22]:
# Grafico 4: Precision vs Recall por quadrante (com curvas iso-F1)
if not pr_df.empty:
    pr_plot = pr_df.copy()
    pr_plot['tipo']  = pr_plot['scenario'].map(stype)
    pr_plot['label'] = pr_plot['scenario'].map(slabel)

    # Mapa quadrante → coluna (1-4)
    q_col = {q: i+1 for i, q in enumerate(cfg.QUADRANT_ORDER)}
    shorts = [cfg.QUADRANT_SHORT[q] for q in cfg.QUADRANT_ORDER]

    fig_pr = make_subplots(
        rows=1, cols=len(cfg.QUADRANT_ORDER),
        subplot_titles=shorts,
        shared_xaxes=False, shared_yaxes=False,
    )

    # Curvas iso-F1
    r_iso = np.linspace(0.05, 1.0, 80)
    for f1_val, dash in [(0.3, 'dot'), (0.5, 'dash'), (0.7, 'dashdot')]:
        p_iso = f1_val * r_iso / (2 * r_iso - f1_val)
        valid = (p_iso > 0) & (p_iso <= 1)
        for col_i in range(1, len(cfg.QUADRANT_ORDER) + 1):
            fig_pr.add_trace(
                go.Scatter(
                    x=r_iso[valid], y=p_iso[valid], mode='lines',
                    line=dict(color='rgba(150,150,150,0.4)', dash=dash, width=1),
                    showlegend=False, hoverinfo='skip',
                    name=f'F1={f1_val}',
                ),
                row=1, col=col_i,
            )

    # Pontos por cenário
    shown_in_legend = set()
    for qi, q in enumerate(cfg.QUADRANT_ORDER):
        sub = pr_plot[pr_plot['quadrant'] == q]
        for tipo in sub['tipo'].unique():
            ss = sub[sub['tipo'] == tipo]
            color = SCENARIO_TYPE_COLORS.get(tipo, '#9E9E9E')
            show_leg = tipo not in shown_in_legend
            if show_leg: shown_in_legend.add(tipo)
            fig_pr.add_trace(
                go.Scatter(
                    x=ss['positive_recall_mean'], y=ss['positive_precision_mean'],
                    mode='markers', name=tipo,
                    marker=dict(color=color, size=7, opacity=0.75),
                    showlegend=show_leg,
                    customdata=ss[['label','model','positive_f1_mean']].values,
                    hovertemplate=(
                        '<b>%{customdata[0]}</b> | %{customdata[1]}<br>'
                        'Recall=%{x:.3f}  Precision=%{y:.3f}<br>'
                        'F1=%{customdata[2]:.3f}<extra></extra>'
                    ),
                ),
                row=1, col=qi+1,
            )

    fig_pr.update_xaxes(range=[0, 1.05], title_text='Recall')
    fig_pr.update_yaxes(range=[0, 1.05], title_text='Precision')
    fig_pr.update_layout(
        title='Precision vs Recall por Quadrante e Cenario',
        height=450, legend_title='Tipo de Cenario',
        margin=dict(l=10, r=10, t=80, b=10),
    )

    save_plot(fig_pr, 'fig_precision_recall_por_quadrante', cfg)

Salvo: fig_precision_recall_por_quadrante.html


In [23]:
# Grafico 5: Matrizes de confusao binarias (OOF, normalizada por linha)
if not cm_df.empty:
    scen_labels = ['openSMILE', 'Best FP', 'Best Fusion']
    n_q = len(cfg.QUADRANT_ORDER)
    shorts = [cfg.QUADRANT_SHORT[q] for q in cfg.QUADRANT_ORDER]

    fig_cm = make_subplots(
        rows=n_q, cols=len(scen_labels),
        subplot_titles=[
            f'{cfg.QUADRANT_SHORT[q]} | {sl}'
            for q in cfg.QUADRANT_ORDER for sl in scen_labels
        ],
        vertical_spacing=0.08, horizontal_spacing=0.06,
    )

    for qi, q in enumerate(cfg.QUADRANT_ORDER):
        for si, sl in enumerate(scen_labels):
            row_data = cm_df[(cm_df['quadrant'] == q) & (cm_df['scenario_label'] == sl)]
            if row_data.empty:
                continue
            r = row_data.iloc[0]
            cm_arr = np.array([[r['TN'], r['FP']], [r['FN'], r['TP']]], dtype=float)
            row_sum = cm_arr.sum(axis=1, keepdims=True)
            cm_norm = cm_arr / np.where(row_sum == 0, 1, row_sum)
            tick_labels = ['Neg.', 'Pos.']
            fig_cm.add_trace(
                go.Heatmap(
                    z=cm_norm, x=tick_labels, y=tick_labels,
                    text=[[f'{cm_norm[i,j]:.2f}' for j in range(2)] for i in range(2)],
                    texttemplate='%{text}',
                    colorscale='Blues', zmin=0, zmax=1,
                    showscale=(qi == 0 and si == len(scen_labels)-1),
                    hovertemplate='Real: %{y}<br>Predito: %{x}<br>Taxa: %{z:.2f}<extra></extra>',
                ),
                row=qi+1, col=si+1,
            )

    fig_cm.update_layout(
        title='Matrizes de Confusao Binarias (OOF, normalizada por linha)',
        height=220 * n_q + 80,
        margin=dict(l=10, r=10, t=80, b=10),
    )
    fig_cm.update_xaxes(title_text='Predito')
    fig_cm.update_yaxes(title_text='Real')

    save_plot(fig_cm, 'fig_confusion_matrices_binary', cfg)

Salvo: fig_confusion_matrices_binary.html


In [24]:
# Grafico 6: Importancia por origem (cenario 7 — OS+AllFP) por quadrante
if not imp_by_origin.empty:
    sub7 = imp_by_origin[
        (imp_by_origin['scenario'] == '7_os_allfp') &
        (imp_by_origin['model'].isin(['ExtraTrees','RandomForest','GradientBoosting','HistGradientBoosting']))
    ]
    if sub7.empty:
        sub7 = imp_by_origin[imp_by_origin['scenario'] == '7_os_allfp']

    if not sub7.empty:
        grp_imp = (
            sub7.groupby(['quadrant','origin'])['importance_pct']
            .mean().reset_index()
        )
        grp_imp['short'] = grp_imp['quadrant'].map(cfg.QUADRANT_SHORT)

        fig_imp = px.bar(
            grp_imp.sort_values('importance_pct', ascending=False),
            x='origin', y='importance_pct',
            color='origin', facet_col='short',
            text=grp_imp.sort_values('importance_pct', ascending=False)['importance_pct'].map('{:.1f}%'.format),
            hover_data={'importance_pct': ':.2f'},
            color_discrete_map=ORIGIN_COLORS,
            title='Importancia por Origem — OS+AllFP (ExtraTrees + RF + GradientBoosting)',
            labels={'importance_pct': 'Importancia relativa (%)', 'origin': 'Origem'},
        )
        fig_imp.update_traces(textposition='outside', showlegend=False)
        fig_imp.update_layout(
            height=420, margin=dict(l=10, r=10, t=60, b=10),
        )
        fig_imp.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
        fig_imp.update_yaxes(matches=None)

        save_plot(fig_imp, 'fig_importancia_por_origem', cfg)

Salvo: fig_importancia_por_origem.html


In [25]:
# Graficos 7a-7d: Top 20 features por quadrante (cenario OS+AllFP)
for q, top20 in top20_by_q.items():
    short = cfg.QUADRANT_SHORT[q]
    top20_plot = top20.copy()
    top20_plot['short_feat'] = (
        top20_plot['feature']
        .str.replace('opensmile__', 'os:', regex=False)
        .str.replace('fprank__', 'fpr:', regex=False)
        .str.replace('fpband__', 'fpb:', regex=False)
        .str.replace('rawpeaks_rank__', 'rpr:', regex=False)
        .str.replace('rawpeaks_band__', 'rpb:', regex=False)
        .str[:60]
    )

    fig_top = px.bar(
        top20_plot.sort_values('importance', ascending=True),
        x='importance', y='short_feat',
        color='origin', orientation='h',
        text=top20_plot.sort_values('importance', ascending=True)['importance'].map('{:.4f}'.format),
        hover_data={'feature': True, 'importance': ':.5f', 'origin': True, 'short_feat': False},
        color_discrete_map=ORIGIN_COLORS,
        title=f'Top 20 Features — {short} (OS+AllFP)',
        labels={'importance': 'Importancia media', 'short_feat': '', 'origin': 'Origem'},
    )
    fig_top.update_traces(textposition='outside')
    fig_top.update_layout(
        height=600, legend_title='Origem',
        margin=dict(l=10, r=80, t=60, b=10),
    )

    save_plot(fig_top, f'fig_top20_features_{short}', cfg)

Salvo: fig_top20_features_Q1.html


Salvo: fig_top20_features_Q2.html


Salvo: fig_top20_features_Q3.html


Salvo: fig_top20_features_Q4.html


In [26]:
# Grafico 8: Visao geral multi-metrica por cenario e quadrante
metrics_cfg_m = [
    ('positive_f1_mean',          'F1 Positivo',   'positive_f1_std'),
    ('balanced_accuracy_mean',     'Balanced Acc',  'balanced_accuracy_std'),
    ('roc_auc_mean',               'ROC AUC',       'roc_auc_std'),
]

multi_frames = []
for mc, ml, sc in metrics_cfg_m:
    if mc not in best_by_scen_q.columns: continue
    sub = best_by_scen_q[['scenario','quadrant','best_model','n_features', mc]].copy()
    sub['valor_std'] = best_by_scen_q[sc] if sc in best_by_scen_q.columns else 0.0
    sub = sub.rename(columns={mc: 'valor'})
    sub['metrica'] = ml
    sub['label']   = sub['scenario'].map(slabel)
    sub['tipo']    = sub['scenario'].map(stype)
    sub['short']   = sub['quadrant'].map(cfg.QUADRANT_SHORT)
    multi_frames.append(sub)

if multi_frames:
    multi_df = pd.concat(multi_frames, ignore_index=True)
    fig_multi = px.bar(
        multi_df.sort_values('scenario'),
        x='label', y='valor', error_y='valor_std',
        color='tipo', facet_col='metrica', facet_row='short',
        barmode='group',
        text=multi_df.sort_values('scenario')['valor'].map('{:.3f}'.format),
        hover_data={'best_model': True, 'n_features': True, 'tipo': False, 'valor_std': ':.4f'},
        color_discrete_map=SCENARIO_TYPE_COLORS,
        title='Visao Geral Multi-Metrica por Cenario e Quadrante',
        labels={'valor': 'Valor', 'label': 'Cenario', 'tipo': 'Tipo', 'short': 'Quadrante'},
    )
    fig_multi.update_traces(
        texttemplate='%{text}', textposition='outside',
        textfont_size=7, showlegend=True,
    )
    fig_multi.update_layout(
        height=1100, legend_title='Tipo de Cenario',
        xaxis_tickangle=-40, margin=dict(l=10, r=10, t=100, b=60),
    )
    fig_multi.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig_multi.update_yaxes(matches=None)

    save_plot(fig_multi, 'fig_multimetrica_por_quadrante', cfg)

Salvo: fig_multimetrica_por_quadrante.html


## 13. Exportação dos resultados

In [27]:
out = cfg.output_dir()

fold_df.to_csv(out / 'fusion_binary_fold_results.csv', index=False)
agg_df.to_csv(out / 'fusion_binary_all_results.csv', index=False)
best_by_q.to_csv(out / 'fusion_binary_best_by_quadrant.csv', index=False)

if not delta_df.empty:
    delta_df.to_csv(out / 'fusion_binary_delta_vs_opensmile.csv', index=False)

class_balance.to_csv(out / 'fusion_binary_class_balance_by_quadrant.csv', index=False)

if not pr_df.empty:
    pr_df.to_csv(out / 'fusion_binary_precision_recall_by_quadrant.csv', index=False)

if not cm_df.empty:
    cm_df.to_csv(out / 'fusion_binary_confusion_matrices.csv', index=False)

if not importance_df.empty:
    importance_df.to_csv(out / 'fusion_binary_feature_importance.csv', index=False)
if not imp_by_origin.empty:
    imp_by_origin.to_csv(out / 'fusion_binary_importance_by_origin.csv', index=False)

# kfold vs holdout placeholder (holdout pode ser implementado opcionalmente)
pd.DataFrame([{'note': 'Validação oficial: GroupKFold(n=5) por song_id. Holdout não executado.'}]).to_csv(
    out / 'fusion_binary_kfold_vs_holdout_comparison.csv', index=False)

print('Exportado para:', out)
for f in sorted(out.glob('*.csv')):
    print(f'  {f.name}')

print('Figuras HTML:', cfg.figures_dir())
for f in sorted(cfg.figures_dir().glob('*.html')):
    print(f'  {f.name}')

Exportado para: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_binary_by_quadrant_validation
  fusion_binary_all_results.csv
  fusion_binary_best_by_quadrant.csv
  fusion_binary_class_balance_by_quadrant.csv
  fusion_binary_confusion_matrices.csv
  fusion_binary_delta_vs_opensmile.csv
  fusion_binary_feature_importance.csv
  fusion_binary_fold_results.csv
  fusion_binary_importance_by_origin.csv
  fusion_binary_kfold_vs_holdout_comparison.csv
  fusion_binary_precision_recall_by_quadrant.csv
Figuras HTML: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_binary_by_quadrant_validation\figures
  fig_balanced_accuracy_por_quadrante.html
  fig_confusion_matrices_binary.html
  fig_delta_positive_f1.html
  fig_importancia_por_origem.html
  fig_multimetrica_por_quadrante.html
  fig_positive_f1_por_cenario_quadrante.html
  fig_precision_recall_por_quadrante.html
  fig_top20_features_Q1.html
  fig_top20_features_Q2.html
  fig_top20_features_Q3.html
  fig_top20_features_Q4.html


## 14. Síntese final

In [28]:
def sintetizar_binario(best_by_q, delta_df, imp_by_origin, top20_by_q, class_balance, cfg) -> str:
    lines = [
        '# Síntese: Fusão openSMILE + Fingerprints — Classificação Binária por Quadrante',
        '',
        '> Síntese gerada automaticamente com base na validação GroupKFold(n=5) por song_id.',
        '> Cada quadrante foi tratado como classe positiva num problema binário one-vs-rest.',
        '> A conclusão é neutra, baseada nos resultados observados, sem hipótese prévia de melhora.',
        '',
        '## Contexto: balanço das classes positivas',
    ]
    for _, r in class_balance.iterrows():
        lines.append(f'  - **{cfg.QUADRANT_SHORT[r["quadrant"]]}**: {r["n_positive"]:,} positivos / {r["n_negative"]:,} negativos ({r["positive_rate"]:.1%})')
    lines.append('')

    # 1. Melhora em algum quadrante?
    lines.append('## 1. O fingerprint melhora a detecção de algum quadrante específico?')
    if delta_df.empty:
        lines.append('_Sem dados de comparação._')
    else:
        for q in cfg.QUADRANT_ORDER:
            short = cfg.QUADRANT_SHORT[q]
            sub = delta_df[delta_df['quadrant'] == q].sort_values('delta_positive_f1', ascending=False)
            if sub.empty: continue
            best_r = sub.iloc[0]
            ganhos = sub['ganho'].value_counts()
            n_melhora = ganhos.drop('sem melhora relevante', errors='ignore').sum()
            total = len(sub)
            lines.append(f'  - **{short}**: melhor Δ F1 = {best_r["delta_positive_f1"]:+.4f} '
                         f'(cenário {slabel(best_r["scenario"])}, {best_r["best_model"]}) | '
                         f'{n_melhora}/{total} cenários com melhora')
    lines.append('')

    # 2. Em quais quadrantes houve ganho real em positive_f1?
    lines.append('## 2. Em quais quadrantes houve ganho real em positive_f1?')
    gained_q, no_gain_q = [], []
    if not delta_df.empty:
        for q in cfg.QUADRANT_ORDER:
            sub = delta_df[delta_df['quadrant'] == q]
            best_d = sub['delta_positive_f1'].max() if not sub.empty else 0.0
            short = cfg.QUADRANT_SHORT[q]
            if best_d > 0.01:
                gained_q.append(f'{short} (Δ {best_d:+.3f})')
            elif best_d > 0.003:
                gained_q.append(f'{short} (Δ {best_d:+.3f} — melhora pequena)')
            else:
                no_gain_q.append(f'{short} (Δ {best_d:+.3f})')
    if gained_q:
        lines.append(f'  - **Com ganho:** {" | ".join(gained_q)}')
    if no_gain_q:
        lines.append(f'  - **Sem ganho:** {" | ".join(no_gain_q)}')
    lines.append('')

    # 3. Melhora de recall
    lines.append('## 3. Em quais quadrantes houve melhora de recall, mesmo com possível queda de precision?')
    if not delta_df.empty:
        for q in cfg.QUADRANT_ORDER:
            short = cfg.QUADRANT_SHORT[q]
            sub = delta_df[delta_df['quadrant'] == q]
            if sub.empty: continue
            best_rec = sub.sort_values('delta_positive_recall', ascending=False).iloc[0]
            d_rec = best_rec['delta_positive_recall']
            d_prec = best_rec.get('delta_positive_precision', np.nan)
            if d_rec > 0.01:
                note = ' (com queda de precision)' if (not np.isnan(d_prec) and d_prec < -0.005) else ''
                d_prec_str = f'{d_prec:+.3f}' if not np.isnan(d_prec) else 'N/A'
                lines.append(f'  - **{short}**: Δ recall = {d_rec:+.3f}, Δ precision = {d_prec_str}{note}')
            else:
                lines.append(f'  - **{short}**: sem ganho expressivo de recall (Δ {d_rec:+.3f})')
    lines.append('')

    # 4. FP Rank vs FP Band
    lines.append('## 4. O ranking geral ou fingerprint por bandas contribuiu mais?')
    if not best_by_q.empty:
        rank_wins, band_wins, tie = 0, 0, 0
        for q in cfg.QUADRANT_ORDER:
            sub = best_by_q[(best_by_q['quadrant'] == q)]
            f1_rank = agg_df[(agg_df['quadrant']==q) & (agg_df['scenario']=='2_fprank_only')]['positive_f1_mean'].max() if len(agg_df) else np.nan
            f1_band = agg_df[(agg_df['quadrant']==q) & (agg_df['scenario']=='3_fpband_only')]['positive_f1_mean'].max() if len(agg_df) else np.nan
            short = cfg.QUADRANT_SHORT[q]
            if np.isnan(f1_rank) or np.isnan(f1_band): tie += 1; continue
            if f1_rank > f1_band + 0.005: rank_wins += 1; lines.append(f'  - {short}: Rank ({f1_rank:.3f}) > Band ({f1_band:.3f})')
            elif f1_band > f1_rank + 0.005: band_wins += 1; lines.append(f'  - {short}: Band ({f1_band:.3f}) > Rank ({f1_rank:.3f})')
            else: tie += 1; lines.append(f'  - {short}: Rank ≈ Band ({f1_rank:.3f} vs {f1_band:.3f})')
        if rank_wins > band_wins:
            lines.append(f'  → **Fingerprint Rank domina** em {rank_wins} quadrante(s).')
        elif band_wins > rank_wins:
            lines.append(f'  → **Fingerprint Band domina** em {band_wins} quadrante(s).')
        else:
            lines.append('  → Nenhum tipo de fingerprint domina sistematicamente.')
    lines.append('')

    # 5. Consistência da fusão
    lines.append('## 5. A fusão melhora o openSMILE de forma consistente ou apenas pontual?')
    if not delta_df.empty:
        total_rows = len(delta_df)
        n_cons = (delta_df['ganho'] == 'melhora consistente').sum()
        n_f1   = (delta_df['ganho'] == 'melhora em F1 positivo').sum()
        n_rec  = (delta_df['ganho'] == 'melhora em recall positivo').sum()
        n_peq  = (delta_df['ganho'] == 'melhora pequena').sum()
        n_sem  = (delta_df['ganho'] == 'sem melhora relevante').sum()
        lines.append(f'  - {total_rows} combinações cenário × quadrante avaliadas.')
        lines.append(f'  - Melhora consistente: {n_cons} | Melhora F1: {n_f1} | Melhora recall: {n_rec} | Pequena: {n_peq} | Sem melhora: {n_sem}')
        if n_cons + n_f1 >= total_rows * 0.4:
            lines.append('  - **Sim, a fusão melhora de forma relativamente consistente** em ≥ 40% das combinações.')
        else:
            lines.append('  - **A melhora é pontual:** ocorre em menos de 40% das combinações testadas.')
    lines.append('')

    # 6. Importância
    lines.append('## 6. As features de fingerprint aparecem entre as mais importantes?')
    if not imp_by_origin.empty:
        sub7 = imp_by_origin[imp_by_origin['scenario'] == '7_os_allfp']
        if not sub7.empty:
            grp_all = sub7.groupby('origin')['importance_pct'].mean().sort_values(ascending=False)
            for o, pct in grp_all.items():
                lines.append(f'  - {o}: {pct:.1f}% (média entre quadrantes e modelos)')
            fp_total = grp_all.drop('openSMILE', errors='ignore').sum()
            lines.append(f'  → Fingerprints somam {fp_total:.1f}% da importância no cenário OS+AllFP.')
        # per quadrant
        for q in cfg.QUADRANT_ORDER:
            short = cfg.QUADRANT_SHORT[q]
            sub = sub7[sub7['quadrant'] == q]
            if sub.empty: continue
            grp = sub.groupby('origin')['importance_pct'].mean().sort_values(ascending=False)
            fp_pct = grp.drop('openSMILE', errors='ignore').sum()
            lines.append(f'  - {short}: fingerprints = {fp_pct:.1f}% da importância')
    lines.append('')

    # 7. Complementaridade ou redundância
    lines.append('## 7. O resultado sugere complementaridade ou redundância?')
    if not delta_df.empty:
        best_global_d = delta_df['delta_positive_f1'].max()
        worst_global_d = delta_df['delta_positive_f1'].min()
        if best_global_d > 0.015:
            lines.append(f'  - **Complementaridade em pelo menos um quadrante**: Δ F1 máximo = {best_global_d:+.4f}.')
        if worst_global_d < -0.01:
            lines.append(f'  - **Redundância/ruído em algum quadrante**: Δ F1 mínimo = {worst_global_d:+.4f}.')
        if abs(best_global_d) < 0.01:
            lines.append('  - Resultado essencialmente neutro: os fingerprints não adicionam informação relevante.')
    lines.append('')

    # 8. Conclusão para o TCC
    lines.append('## 8. Conclusão para a análise binária por quadrante no TCC')
    if not delta_df.empty:
        q_with_gain = [cfg.QUADRANT_SHORT[q] for q in cfg.QUADRANT_ORDER
                       if delta_df[delta_df['quadrant']==q]['delta_positive_f1'].max() > 0.005]
        q_without = [cfg.QUADRANT_SHORT[q] for q in cfg.QUADRANT_ORDER if cfg.QUADRANT_SHORT[q] not in q_with_gain]
        if q_with_gain:
            lines.append(f'  - Os fingerprints acústicos trouxeram ganho em F1 positivo para: **{" | ".join(q_with_gain)}**.')
        if q_without:
            lines.append(f'  - Não houve ganho relevante para: {" | ".join(q_without)}.')
        lines.append('  - O resultado valida a investigação de fusão, ainda que o ganho não seja universal.')
        lines.append('  - Recomenda-se reportar os resultados por quadrante individualmente, pois a fusão não impacta uniformemente todos os quadrantes.')
        lines.append('  - O protocolo GroupKFold por song_id garante validade metodológica das comparações.')
    lines += [
        '',
        '---',
        f'_Gerado em: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")} | Validação: GroupKFold(n=5) por song_id | Métrica principal: positive_f1_',
    ]
    return '\n'.join(lines)


synthesis = sintetizar_binario(best_by_q, delta_df, imp_by_origin, top20_by_q, class_balance, cfg)
display(Markdown(synthesis))

with open(cfg.output_dir() / 'fusion_binary_interpretation_summary.md', 'w', encoding='utf-8') as f:
    f.write(synthesis)
print('Síntese exportada:', cfg.output_dir() / 'fusion_binary_interpretation_summary.md')

# Síntese: Fusão openSMILE + Fingerprints — Classificação Binária por Quadrante

> Síntese gerada automaticamente com base na validação GroupKFold(n=5) por song_id.
> Cada quadrante foi tratado como classe positiva num problema binário one-vs-rest.
> A conclusão é neutra, baseada nos resultados observados, sem hipótese prévia de melhora.

## Contexto: balanço das classes positivas
  - **Q1**: 3,217 positivos / 3,289 negativos (49.4%)
  - **Q2**: 1,019 positivos / 5,487 negativos (15.7%)
  - **Q3**: 1,382 positivos / 5,124 negativos (21.2%)
  - **Q4**: 888 positivos / 5,618 negativos (13.6%)

## 1. O fingerprint melhora a detecção de algum quadrante específico?
  - **Q1**: melhor Δ F1 = +0.0008 (cenário 5. OS+FPRank, ExtraTrees) | 0/9 cenários com melhora
  - **Q2**: melhor Δ F1 = +0.0005 (cenário 10. OS+AllFP+Raw, LogisticRegression) | 0/9 cenários com melhora
  - **Q3**: melhor Δ F1 = +0.0102 (cenário 7. OS+AllFP, RidgeClassifier) | 5/9 cenários com melhora
  - **Q4**: melhor Δ F1 = +0.0109 (cenário 9. OS+RawPks, LogisticRegression) | 2/9 cenários com melhora

## 2. Em quais quadrantes houve ganho real em positive_f1?
  - **Com ganho:** Q3 (Δ +0.010) | Q4 (Δ +0.011)
  - **Sem ganho:** Q1 (Δ +0.001) | Q2 (Δ +0.001)

## 3. Em quais quadrantes houve melhora de recall, mesmo com possível queda de precision?
  - **Q1**: sem ganho expressivo de recall (Δ +0.007)
  - **Q2**: sem ganho expressivo de recall (Δ -0.001)
  - **Q3**: Δ recall = +0.026, Δ precision = -0.002
  - **Q4**: sem ganho expressivo de recall (Δ +0.004)

## 4. O ranking geral ou fingerprint por bandas contribuiu mais?
  - Q1: Rank (0.721) > Band (0.706)
  - Q2: Rank (0.356) > Band (0.347)
  - Q3: Rank ≈ Band (0.544 vs 0.543)
  - Q4: Band (0.351) > Rank (0.333)
  → **Fingerprint Rank domina** em 2 quadrante(s).

## 5. A fusão melhora o openSMILE de forma consistente ou apenas pontual?
  - 36 combinações cenário × quadrante avaliadas.
  - Melhora consistente: 0 | Melhora F1: 2 | Melhora recall: 1 | Pequena: 4 | Sem melhora: 29
  - **A melhora é pontual:** ocorre em menos de 40% das combinações testadas.

## 6. As features de fingerprint aparecem entre as mais importantes?
  - openSMILE: 54.4% (média entre quadrantes e modelos)
  - Fingerprint Band: 30.6% (média entre quadrantes e modelos)
  - Fingerprint Rank: 15.1% (média entre quadrantes e modelos)
  → Fingerprints somam 45.6% da importância no cenário OS+AllFP.
  - Q1: fingerprints = 38.9% da importância
  - Q2: fingerprints = 45.8% da importância
  - Q3: fingerprints = 47.2% da importância
  - Q4: fingerprints = 50.7% da importância

## 7. O resultado sugere complementaridade ou redundância?
  - **Redundância/ruído em algum quadrante**: Δ F1 mínimo = -0.0678.

## 8. Conclusão para a análise binária por quadrante no TCC
  - Os fingerprints acústicos trouxeram ganho em F1 positivo para: **Q3 | Q4**.
  - Não houve ganho relevante para: Q1 | Q2.
  - O resultado valida a investigação de fusão, ainda que o ganho não seja universal.
  - Recomenda-se reportar os resultados por quadrante individualmente, pois a fusão não impacta uniformemente todos os quadrantes.
  - O protocolo GroupKFold por song_id garante validade metodológica das comparações.

---
_Gerado em: 2026-06-09 09:42 | Validação: GroupKFold(n=5) por song_id | Métrica principal: positive_f1_

Síntese exportada: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_binary_by_quadrant_validation\fusion_binary_interpretation_summary.md
